# 4 The Limits of Context Stuffing & Why Retrieval (RAG) Fixes them


From Documents to Usable Context: Adding context without changing the model

Purpose of this section

>Help participants internalize that getting documents into text is not enough. Value only appears once documents become retrievable, bounded, and evaluable context that a model can reason over.

This is where we move from document plumbing to AI system behavior.

Prerequisites
* Section 2 complete (API key and endpoint configured)


The document we'll be working with `Basic-Fantasy-RPG-Rules-r142.md` has already been processed and is included in the repo.

## 4.1 The Real Problem: "We Have Text. Now What?"

After extraction, teams often experience a brief sense of victory.

> "The document is in Markdown. We're done."

But text alone does not create intelligence. 

A common first instinct is the *"just add it to context"* approach: take the entire document, paste it into the prompt, ask the model a question.

It feels reasonable. The information is there. The model can read. What could go wrong?

Let’s test that assumption.

Before we send an entire document to a model, we need to understand a critical constraint: models don’t process text — they process tokens. Context windows are measured in tokens, not characters or words.

### 4.1.1 Understanding Tokens: The Unit That Actually Matters

When we talk about large language models, the unit that actually matters isn’t the “model size;” it’s the token count. A token isn’t a word; it’s a chunk of text the model processes as a single unit. Sometimes that’s a full word, sometimes part of a word, sometimes punctuation or formatting. 

On average, a token is about three-quarters of a word in English, but formatting and structure can dramatically increase the count. Every system instruction, user prompt, retrieved document, and model response consumes tokens. That means tokens define your context window — what the model can “see” at once — and they define your budget. If you exceed the limit, something gets truncated. In production systems, that’s where subtle failures begin.

Tokens aren’t just about cost — they’re about attention. The model distributes attention across every token in context, so irrelevant text, broken formatting, repeated headers, and layout noise dilute the signal.
Dumping raw document extractions into a prompt wastes capacity and weakens performance. Clean structure, reduced noise, and preserved hierarchy make the semantic signal denser — and dense signal is what models perform best on. 

So when you hear “tokens,” don’t think merely of a billing metric. 
* Think capacity. 
* Think attention. 
* Think signal-to-noise ratio.

To make this concrete and get metrics, we’ll use tiktoken, OpenAI’s tokenizer library. It lets us see how many tokens a given piece of text would consume for a specific model. Instead of guessing, we can measure. That visibility changes how you think about prompts. It turns token management from an abstract constraint into something you can actively design around.

And tiktoken isn’t the only option. There are multiple token visualizers available online that let you experiment interactively. The important point isn’t the tool or the exact number, it’s the awareness and rough order of magnitude. Once you start thinking in terms of tokens, you start making different architectural decisions.

We’re about to measure how expensive “just add the whole document” really is.

We start by installing `tiktoken`.


In [1]:
! pip install tiktoken -q

In the next cell, load `Basic-Fantasy-RPG-Rules-r142.md` and count its tokens. The result will tell us whether our naive approach survives first contact with a real context window.


In [2]:
import tiktoken

# Load your markdown file
with open("./Basic-Fantasy-RPG-Rules-r142.md", "r", encoding="utf-8") as f:
    text = f.read()

# Choose model encoding
encoding = tiktoken.get_encoding("cl100k_base")

# If you're encoding for gpt-4o
# encoding = tiktoken.encoding_for_model("gpt-4o")  # or "gpt-4"

tokens = encoding.encode(text)

print(f"Number of tokens: {len(tokens)}")

Number of tokens: 205443


The number of tokens is roughly `205,000`, well within the 4 million context window of the `granite-3-2-8b-instruct` model. 

In earlier generations of models, this approach would often fail outright due to context window limits. As context windows expand into the millions of tokens, that hard constraint is becoming less common.

What does not disappear, however, are the economic and performance tradeoffs. Larger windows still incur higher latency, higher cost, and reduced focus as irrelevant tokens compete for attention. Those constraints are architectural and they still apply.



### 4.1.2 Testing the Naive Approach: Just Paste the Whole Document

But this is a lab. So rather than speculate, let’s try it. The question now isn’t whether it fits; it’s whether this is the right way to use that capacity.

Run the following code and see what happens.


In [3]:
# Load MaaS Configuration
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base
url_models = f"{endpoint_base}/models"
url_chat = f"{endpoint_base}/chat/completions"

import requests

# Load markdown file
with open("Basic-Fantasy-RPG-Rules-r142.md", "r", encoding="utf-8") as f:
    document_text = f.read()

headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}

data = {
    "model": "granite-3-2-8b-instruct",
    "messages": [
        {
            "role": "system",
            "content": "Answer the question using only the provided document."
        },
        {
            "role": "user",
            "content": f"""
You are given the following document:

{document_text}

Question:
What happens if a Thief fails an Open Locks attempt?
"""
        }
    ]
}

response = requests.post(url_chat, headers=headers, json=data)

print(response.json())


{'error': {'message': "litellm.ContextWindowExceededError: litellm.BadRequestError: ContextWindowExceededError: OpenAIException - This model's maximum context length is 120000 tokens. However, you requested 241974 tokens in the messages, Please reduce the length of the messages. None\nmodel=granite-3-2-8b-instruct. context_window_fallbacks=None. fallbacks=None.\n\nSet 'context_window_fallback' - https://docs.litellm.ai/docs/routing#fallbacks. Received Model Group=granite-3-2-8b-instruct\nAvailable Model Group Fallbacks=None", 'type': None, 'param': None, 'code': '400'}}


The error that returns comes as a bit of a surprise given that **120,000 is quite a bit less than 4 million.** 

What’s going on here?


We didn’t do anything wrong. We asked the system to process more tokens than the serving stack allows. 

Even if a model can support huge context in theory, **our deployed endpoint enforces 120k**.  

Practical context length is a deployment decision. “Just paste the whole doc” fails for at least one of: context limits, cost, latency, or focus. 


### 4.1.3 Why servers limit token counts
It’s tempting to assume context limits are arbitrary. They’re not.

Every additional token increases:
* Memory pressure — Attention mechanisms scale with context length. More tokens means more GPU memory consumption.
* Compute time — Attention isn’t free. The longer the context, the more work required per forward pass.
* Latency variance — Large inputs can create tail latency issues, which matter in multi-tenant systems.
* Throughput constraints — If one request monopolizes GPU memory, fewer concurrent users can be served.
* Cost exposure — Longer contexts burn more compute, which directly translates to dollars.

From a research perspective, maximizing context length is a capability milestone. From an operations perspective, it’s a resource management problem.

Serving stacks place limits because they need predictable performance, bounded memory usage, and fair allocation across customers. A 120k cap isn’t arbitrary — it’s a tradeoff between capability and reliability.

Here it fails immediately in context.  The document fits in a model. It doesn’t fit in this system. So our job isn’t “get text.” Our job is to “get the right text.” That’s retrieval. 

But, since this is a lab, let’s see if we can make it work around this unexpected constraint to see what other issues pop up.

### 4.1.4 Measuring the Token Cost Before We Send

Before we attempt to send the full document to the model, let's find out exactly what we're asking the system to handle. This cell loads the Markdown file we produced with Docling, encodes both the document and our test question into tokens, and prints the totals. 

The numbers will tell us whether we're within budget or whether we're about to hit that 120k token wall again.



In [12]:
import requests
import tiktoken
import json

source_md = "Basic-Fantasy-RPG-Rules-r142.md"

with open(source_md, "r", encoding="utf-8") as f:
    document_text = f.read()

question = "What happens if a Thief fails an Open Locks attempt?"

# Token estimate (good enough for a lab)
enc = tiktoken.get_encoding("cl100k_base")
doc_tokens = len(enc.encode(document_text))
question_tokens = len(enc.encode(question))

print ("a Quick Check of Tokens:")
print(f"Document tokens (estimate): {doc_tokens:,}")
print(f"Question tokens (estimate):  {question_tokens:,}")
print(f"Total prompt tokens (estimate): {(doc_tokens + question_tokens):,}")
print("Expected max context (per error): 120,000 tokens\n")


a Quick Check of Tokens:
Document tokens (estimate): 205,443
Question tokens (estimate):  12
Total prompt tokens (estimate): 205,455
Expected max context (per error): 120,000 tokens



## 4.1.5 Forcing It to Fit: Trimming to a Token Budget

We know the full document exceeds the endpoint's 120,000-token limit. So let's try the brute-force workaround: trim the document to fit. 

This cell sets a hard budget of 100,000 tokens, reserves 2,000 tokens for the system instruction, question, and model response, then truncates the document at the budget boundary. Whatever falls beyond that cutoff is simply discarded. 

We then send the trimmed version to the model and see if we get a usable answer.


In [13]:
MAX_CTX = 100_000
RESERVE = 2_000  # room for system + question + answer
BUDGET = MAX_CTX - RESERVE

tokens = enc.encode(document_text)
document_text_trimmed = enc.decode(tokens[:BUDGET])

print("Trimmed doc tokens:", len(enc.encode(document_text_trimmed)))

data["messages"][1]["content"] = f"""
You are given the following document:

{document_text_trimmed}

Question:
What happens if a Thief fails an Open Locks attempt?
"""
response = requests.post(url_chat, headers=headers, json=data)
print(response.status_code)
print(response.json())

Trimmed doc tokens: 98000
200
{'id': 'chatcmpl-b15bdc5fe4d142adb5675dff65051273', 'created': 1771607716, 'model': 'granite-3-2-8b-instruct', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'If a Thief fails an Open Locks attempt, the spell does not take effect, and any attempt to pick the lock will trigger it as if it were being opened normally.', 'role': 'assistant', 'tool_calls': None, 'function_call': None}, 'provider_specific_fields': {'stop_reason': None}}], 'usage': {'completion_tokens': 37, 'prompt_tokens': 114404, 'total_tokens': 114441, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'service_tier': None, 'prompt_logprobs': None, 'kv_transfer_params': None}


And this time, the answer is the one we were looking for.

>'If a Thief fails an Open Locks attempt, the Thief must wait until they have gained another level of experience before trying again.'

#### 4.1.6 The Effective Context Window: Why Bigger Isn't Always Better

Fitting within the token limit is necessary, but it isn't sufficient.

Models have a *supported* context window -- the hard maximum the architecture allows -- and an *effective* context window, which is the range over which the model actually performs well. These are not the same number.

As context grows, models begin to struggle with what's sometimes called the needle-in-a-haystack problem: relevant information exists in the prompt, but the model fails to find or prioritize it. Attention gets diffused across too many tokens. When multiple documents are present, the model can start mixing context between them, attributing facts from one source to another. The result is answers that are coherent, confident, and wrong in ways that are hard to detect -- which is the most dangerous kind of failure in a production system.

There is also a hardware reality that constrains every deployment decision. A larger context window requires more GPU memory to process. A developer running a small model on a local GPU might handle a 16k context window comfortably. That same setup cannot handle a 1 million token window. In cloud or on-premise deployments, this translates directly to cost: more memory per request, fewer concurrent users, higher infrastructure spend. The context window isn't just a model capability -- it's a resource budget that someone has to pay for.

#### 4.1.7 Why "Make It Fit" Isn't a Strategy

Yes, we can force it to fit. But we're mutilating the input, paying a heavy token cost, waiting longer than necessary, and operating well inside the zone where quality degrades. This approach doesn't scale.

The model doesn't need the whole book. It needs the right paragraph.

What we need is a way to find that right paragraph automatically, without manually searching the document or guessing what the model needs. That's what retrieval does. In the next notebook, we'll build a simple RAG pipeline that replaces the "paste everything" approach with a targeted search: embed the document, index the chunks, retrieve only what's relevant, and hand that to the model instead.


# Continue to `02_RAG_Retrieval.ipynb`